<a href="https://colab.research.google.com/github/andresjaimes2004/tesis-desercion/blob/main/notebooks/05_exportacion_powerbi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Exportación para Power BI (OE4)

Fase 6 de CRISP-DM. Genera el archivo `powerbi/tablero_desercion.xlsx` con todo lo que necesita el tablero: los estudiantes con etiquetas en español, el puntaje de riesgo fuera de muestra en los tres momentos con sus tres factores principales, la calibración del puntaje, las métricas del modelo, la importancia SHAP y la asociación de la Fase 2.

**Tiempo estimado en Colab: entre 3 y 8 minutos.**

In [1]:
# Preparar el entorno (igual que en el cuaderno 00)
import os, sys
REPO_URL = "https://github.com/andresjaimes2004/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"
if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [2]:
import os, warnings
os.environ["PYTHONWARNINGS"] = "ignore:Unknown solver options"
warnings.filterwarnings("ignore", message="Unknown solver options")

import pandas as pd
from src import config, prep, export
from src.data import load_raw, binarize_target
from src.utils import set_seed

set_seed()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
df = prep.add_no_enrollment_flags(binarize_target(load_raw()))
idx_train, idx_test = prep.load_split()

## 1. Generar el archivo para Power BI

Para cada momento, el modelo final puntúa a cada estudiante sin haberlo visto: los del entrenamiento con validación cruzada de 5 pliegues y los de la prueba con el modelo entrenado en todo el entrenamiento. Con el mismo modelo se calculan los valores SHAP y se guardan las tres variables que más suben el riesgo de cada estudiante.

In [3]:
ruta, hojas = export.export_workbook(df, idx_train, idx_test)
print("Archivo:", ruta)
for nombre, t in hojas.items():
    print(f"  {nombre:16s} {t.shape[0]:6d} filas × {t.shape[1]:2d} columnas")

Archivo: /content/tesis-desercion/powerbi/tablero_desercion.xlsx
  Estudiantes        4424 filas × 27 columnas
  Predicciones      13272 filas × 11 columnas
  Calibracion          30 filas ×  8 columnas
  Metricas              3 filas × 13 columnas
  ImportanciaSHAP      31 filas ×  3 columnas
  Asociacion           36 filas ×  8 columnas
  Control              15 filas ×  2 columnas


## 2. Verificación

Las métricas recalculadas a partir de la hoja `Predicciones` (solo prueba) deben coincidir con las de la Fase 5.

In [4]:
pred = hojas["Predicciones"]
prueba = pred[pred["Origen"] == "Prueba"]
for m, g in prueba.groupby("Momento", sort=False):
    alerta = g["Alerta"] == "Sí"
    vp = int((alerta & (g["Desertó"] == 1)).sum())
    print(f"{m:22s} sensibilidad = {vp / g['Desertó'].sum():.3f} | precisión = {vp / alerta.sum():.3f}")
print()
print(hojas["Metricas"][["Momento", "Sensibilidad", "Precisión"]].round(3).to_string(index=False))

Matrícula              sensibilidad = 0.827 | precisión = 0.503
Cierre 1.er semestre   sensibilidad = 0.824 | precisión = 0.716
Cierre 2.º semestre    sensibilidad = 0.845 | precisión = 0.779

             Momento  Sensibilidad  Precisión
           Matrícula         0.827      0.503
Cierre 1.er semestre         0.824      0.716
 Cierre 2.º semestre         0.845      0.779


## 3. Resumen para el documento

Ejecutar esta celda y enviar su salida completa.

In [5]:
print("CONTROL (valores que el tablero debe mostrar):")
print(hojas["Control"].to_string(index=False))
print("\nCALIBRACIÓN DEL PUNTAJE EN PRUEBA (cierre 2.º semestre):")
c = hojas["Calibracion"]
print(c[c["Orden momento"] == 3].sort_values("Orden rango")[["Rango de puntaje", "Estudiantes", "Desertores", "Deserción observada (%)"]].to_string(index=False))
print("\nEJEMPLO DE PREDICCIONES CON ALERTA (cierre 2.º semestre, prueba):")
e = prueba[(prueba["Orden momento"] == 3) & (prueba["Alerta"] == "Sí")].sort_values("Puntaje de riesgo", ascending=False).head(5)
print(e[["ID", "Puntaje de riesgo", "Desertó", "Factor 1", "Factor 2", "Factor 3"]].to_string(index=False))

CONTROL (valores que el tablero debe mostrar):
                                             Indicador  Valor esperado
                                           Estudiantes        4424.000
                                            Desertores        1421.000
                                 Tasa de deserción (%)          32.100
                         Matrícula | alertas en prueba         467.000
           Matrícula | desertores detectados en prueba         235.000
              Cierre 1.er semestre | alertas en prueba         327.000
Cierre 1.er semestre | desertores detectados en prueba         234.000
               Cierre 2.º semestre | alertas en prueba         308.000
 Cierre 2.º semestre | desertores detectados en prueba         240.000
                    Matrícula | sensibilidad en prueba           0.827
                       Matrícula | precisión en prueba           0.503
         Cierre 1.er semestre | sensibilidad en prueba           0.824
            Cierre 1.er semest

## 4. Guardar en GitHub y descargar para Power BI

La siguiente celda empaqueta las carpetas `powerbi/` y `reports/` en `fase6.zip`. Descárgalo y descomprímelo: el archivo `powerbi/tablero_desercion.xlsx` es el que vas a abrir en Power BI. Sube la carpeta `powerbi` completa al repositorio con el mensaje `Fase 6: datos para Power BI`, y guarda este cuaderno con Archivo > Guardar una copia en GitHub.

In [6]:
!cd /content/tesis-desercion && zip -qr /content/fase6.zip powerbi reports && echo "Listo: /content/fase6.zip"

Listo: /content/fase6.zip
